<a href="https://colab.research.google.com/github/Gowtham13042007/cron_job/blob/main/titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [68]:
import pandas as pd

df=pd.read_csv('/content/Titanic-Dataset.csv')
#print(df.shape)
#print(df.dtypes)
df.head()
#print(df.isnull().sum())  # drop Name,Cabin,PassengerId
#df.describe(include='all')
#df.duplicated().sum()
#df['Survived'].value_counts()


# drop Name,Cabin,PassengerId,Ticket
# need to fill the missing values of Age,Embarked

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [69]:
import seaborn as sns
import matplotlib.pyplot as plt
#sns.countplot(x='Survived', data=df)
#sns.countplot(x='Sex', hue='Survived', data=df)
#sns.countplot(x='Pclass', hue='Survived', data=df)
#sns.countplot(x='SibSp', hue='Survived', data=df)
#sns.countplot(x='Parch', hue='Survived', data=df)


In [70]:
df = df[['Name','Survived', 'Pclass', 'Sex', 'Age','SibSp','Parch','Fare','Embarked']]
for i, name in enumerate(df['Name']):
    if 'Mrs.' in name:
        df.loc[i, 'Name'] = 0
    elif 'Miss.' in name:
        df.loc[i, 'Name'] = 1
    elif 'Master.' in name:
        df.loc[i, 'Name'] = 2
    elif 'Mr.' in name:
        df.loc[i, 'Name'] = 3
    else:
        df.loc[i, 'Name'] = 4
df['Name'] = df['Name'].astype(int)
df.head()
print(df.nunique())
print(df.isnull().sum())
df.head()

Name          5
Survived      2
Pclass        3
Sex           2
Age          88
SibSp         7
Parch         7
Fare        248
Embarked      3
dtype: int64
Name          0
Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64


,Name,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,0,3,male,22.0,1,0,7.2500,S
1,0,1,1,female,38.0,1,0,71.2833,C
2,1,1,3,female,26.0,0,0,7.9250,S
3,0,1,1,female,35.0,1,0,53.1000,S
4,3,0,3,male,35.0,0,0,8.0500,S


In [71]:
df['Embarked']=df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Sex']=df['Sex'].map({'male':0,'female':1})
df['Embarked']=df['Embarked'].map({'S':0,'C':1,'Q':2})
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df.head()

,Name,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,IsAlone
0,3,0,3,0,22.0,1,0,7.2500,0,2,0
1,0,1,1,1,38.0,1,0,71.2833,1,2,0
2,1,1,3,1,26.0,0,0,7.9250,0,1,1
3,0,1,1,1,35.0,1,0,53.1000,0,2,0
4,3,0,3,0,35.0,0,0,8.0500,0,1,1


In [72]:
known_age=df[df['Age'].notnull()]
miss_age=df[df['Age'].isnull()]
features = ['Name','Pclass', 'Sex', 'Fare', 'SibSp', 'Fare','Embarked']

from sklearn.ensemble import RandomForestRegressor
model=RandomForestRegressor(n_estimators=100)
X=known_age[features]
y=known_age['Age']
model.fit(X,y)
pred=model.predict(miss_age[features])
print(pred)
df.loc[df['Age'].isnull(), 'Age'] = pred


[42.08566667 31.13803905 30.17       32.96644048 19.82378571 26.97802571
 31.9684     21.6357619  24.64828824 32.18653565 30.46376828 39.10491216
 21.6357619  23.313125   38.2267381  35.85771429  6.88231667 26.97802571
 30.46376828 20.8645119  30.46376828 30.46376828 26.97802571 30.54690023
  4.99283333 30.46376828 44.63994048  3.59908333 30.22       30.96374813
 25.04438961  8.10504762 43.85333333 52.0135      6.09133333 14.24752381
 32.96333333 45.45       29.88       44.63994048 21.6357619  16.22133333
 38.20894769 26.97802571  5.4995     22.14       15.14875     6.209375
 30.96374813 51.67166667 44.63994048 21.6357619  46.8605     21.6357619
 35.18003237 57.49711905 35.85771429 43.3        21.6357619  31.15661797
 25.75395957 30.46376828 28.84733333 16.22133333  4.45691667 36.18
 26.97802571 27.28       51.1645     32.96644048 19.82378571 19.82378571
 39.10491216 30.17       21.6357619  36.37       26.97802571 24.04709596
  5.4995     26.97802571 22.71908009 35.18003237 31.04      

In [73]:
df.isnull().sum()
df.head()

,Name,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,IsAlone
0,3,0,3,0,22.0,1,0,7.2500,0,2,0
1,0,1,1,1,38.0,1,0,71.2833,1,2,0
2,1,1,3,1,26.0,0,0,7.9250,0,1,1
3,0,1,1,1,35.0,1,0,53.1000,0,2,0
4,3,0,3,0,35.0,0,0,8.0500,0,1,1


In [53]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.8 MB/s eta 0:00:00


In [75]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[['Age', 'Fare']] = scaler.fit_transform(df[['Age', 'Fare']])
print(df.dtypes)

X=df.drop('Survived', axis=1)
y=df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
model = XGBClassifier(
    n_estimators=900,
    learning_rate=0.03,
    max_depth=5,            # Dropped from 6 to 4 to reduce overfitting
    subsample=0.9,          # Uses 90% of data per tree to add randomness
    colsample_bytree=0.9,   # Uses 90% of features per tree
    reg_alpha=0.1,          # L1 regularization
    reg_lambda=1.0,         # L2 regularization
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

model = CatBoostClassifier(
    iterations=900,         # CatBoost uses 'iterations' instead of n_estimators
    learning_rate=0.03,     # Good bump up from 0.01!
    depth=5,                # CatBoost uses 'depth' instead of max_depth
    subsample=0.7,
    colsample_bylevel=0.7,  # Changed from colsample_bytree
    loss_function='Logloss',
    random_seed=42,         # CatBoost uses random_seed instead of random_state
    verbose=False
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)



Name            int64
Survived        int64
Pclass          int64
Sex             int64
Age           float64
SibSp           int64
Parch           int64
Fare          float64
Embarked        int64
FamilySize      int64
IsAlone         int64
dtype: object
Accuracy: 0.8379888268156425
Accuracy: 0.8212290502793296


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


df = pd.read_csv('/content/Titanic-Dataset.csv')


X = df.drop('Survived', axis=1)
y = df['Survived']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


def preprocess_data(df):
    df = df.copy()
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col', \
                                'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace(['Mlle', 'Ms'], 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = np.where(df['FamilySize'] > 1, 0, 1)
    return df


X_train_prepped = preprocess_data(X_train_raw)
X_test_prepped = preprocess_data(X_test_raw)

test_passenger_ids = X_test_prepped['PassengerId']

# Drop irrelevant features for modeling
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
X_train_prepped = X_train_prepped.drop(columns=drop_cols)
X_test_prepped = X_test_prepped.drop(columns=drop_cols)

# 3. Define Preprocessing Pipelines
numericala_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']
categorical_cols = ['Sex', 'Title', 'Embarked']
numericalb_cols = ['Pclass', 'SibSp', 'Parch', 'FamilySize', 'IsAlone']


numericala_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('numa', numericala_transformer, numericala_cols),
        ('cat', categorical_transformer, categorical_cols),
        ('numb','passthrough',numericalb_cols)
    ])

# 4. Build Model Pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

# 5. Hyperparameter Tuning
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 5, 10],
    'model__min_samples_split': [2, 5]
}

# Further split train set to create a validation set
X_train, X_val, y_train_split, y_val_split = train_test_split(X_train_prepped, y_train, test_size=0.2, random_state=42)

# GridSearchCV handles CV internally, but training on X_train/y_train_split works perfectly
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train_split)

print(f"Best Parameters: {grid_search.best_params_}")

# 6. Evaluate on Validation Set
best_model = grid_search.best_estimator_
val_predictions = best_model.predict(X_val)

print("\nValidation Model Evaluation:")
print(f"Accuracy: {accuracy_score(y_val_split, val_predictions):.4f}")
print(classification_report(y_val_split, val_predictions))

# 7. Generate Predictions for Holdout Test Set
test_predictions = best_model.predict(X_test_prepped)

output = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Survived': test_predictions
})
output.to_csv('submission.csv', index=False)
print("\nSubmission file 'submission.csv' generated successfully!")

<>:23: SyntaxWarning: invalid escape sequence '\.'
<>:23: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_990/1654581065.py:23: SyntaxWarning: invalid escape sequence '\.'
  df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


Best Parameters: {'model__max_depth': 5, 'model__min_samples_split': 5, 'model__n_estimators': 100}

Validation Model Evaluation:
Accuracy: 0.8392
              precision    recall  f1-score   support

           0       0.86      0.89      0.87        87
           1       0.81      0.77      0.79        56

    accuracy                           0.84       143
   macro avg       0.83      0.83      0.83       143
weighted avg       0.84      0.84      0.84       143


Submission file 'submission.csv' generated successfully!
